In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

In [16]:
df = pd.read_parquet('features.parquet')

In [37]:
df_gmm = df.copy()
df_gmm["datetime"] = pd.to_datetime(df_gmm["datetime"], utc=True)
df_gmm = df_gmm.sort_values("datetime").set_index("datetime")

r = df_gmm["r"]
close = df_gmm["close"]
log_close = np.log(close)

df_gmm["ret_6"] = r.rolling(6).sum()
df_gmm["ret_24"] = r.rolling(24).sum()

df_gmm["rv_42"] = np.sqrt((r ** 2).rolling(42).sum())
df_gmm["rv_72"] = np.sqrt((r ** 2).rolling(72).sum())

df_gmm["down_rv_72"] = np.sqrt((np.minimum(r, 0.0) ** 2).rolling(72).sum())
df_gmm["up_rv_72"] = np.sqrt((np.maximum(r, 0.0) ** 2).rolling(72).sum())

df_gmm["downside_share_72"] = df_gmm["down_rv_72"] / (
    df_gmm["down_rv_72"] + df_gmm["up_rv_72"]
)

df_gmm["rv_ratio_24_72"] = (
    np.sqrt((r ** 2).rolling(24).sum()) / df_gmm["rv_72"]
)

df_gmm["ma_dist_72"] = log_close - log_close.rolling(72).mean()
df_gmm["ma_slope_42"] = log_close.rolling(42).mean().diff(6)
df_gmm["drawdown_180"] = close / close.rolling(180).max() - 1

df_gmm["ret_24_over_rv_72"] = df_gmm["ret_24"] / df_gmm["rv_72"]

feature_cols = [
    "r",
    "roll_skew",
    "ret_6",
    "ret_24",
    "rv_42",
    "rv_72",
    "rv_ratio_24_72",
    "downside_share_72",
    "ma_dist_72",
    "ma_slope_42",
    "drawdown_180",
    "ret_24_over_rv_72",
]

df_gmm = (
    df_gmm[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

X = df_gmm[feature_cols].to_numpy(dtype=float)
Y = np.zeros(len(df_gmm), dtype=int)
dates = df_gmm.index

r_col = feature_cols.index("r")

In [18]:
df_gmm.head()

,r,real_vol,range_pct,vol_ratio,mom,roll_skew
datetime,,,,,,
2020-04-02 08:00:00+00:00,0.005482,0.009154,0.015803,0.939951,0.084590,0.678023
2020-04-02 12:00:00+00:00,0.017889,0.009471,0.030530,1.836026,0.104356,0.672250
2020-04-02 16:00:00+00:00,0.000000,0.008056,0.074671,5.040902,0.141964,0.676163
2020-04-02 20:00:00+00:00,0.000000,0.008040,0.031707,1.195707,0.146042,0.722093
2020-04-03 00:00:00+00:00,-0.002283,0.007612,0.014968,0.663369,0.122949,0.746489


In [38]:
def make_future_return_target(X_all, end_idx, horizon=24, r_col=0):
    y = []

    for i in end_idx:
        future_r = X_all[i + 1 : i + 1 + horizon, r_col]
        y.append(np.sum(future_r))

    return np.array(y, dtype=float)

In [39]:
def walk_forward(X, Y, dates, model, tr, v, te, em, **model_kwargs):
    preds = []
    n = len(Y)
    t = np.arange(n)
    for k in range(0, n - tr - v - 2 * em - te + 1, te):
        train = t[k : k + tr]
        val = t[k + tr + em : k + tr + em + v]
        test = t[k + tr + 2 * em + v : k + tr + 2 * em + v + te]
        X_train = X[train]
        Y_train = Y[train]
        X_val = X[val]
        Y_val = Y[val]
        X_test = X[test]
        Y_test = Y[test]
        Y_hat = model(
            X_train=X_train,
            Y_train=Y_train,
            X_val=X_val,
            Y_val=Y_val,
            X_test=X_test,
            dates=dates,
            X_all=X,
            train_idx=train,
            val_idx=val,
            test_idx=test,
            **model_kwargs,)

        fold = pd.DataFrame(
            {
                "Y_true": Y_test,
                "Y_pred": Y_hat,},
            index=dates[test],)
        preds.append(fold)

    return pd.concat(preds)

In [40]:
def gmm_model(
    X_train,
    Y_train,
    dates,
    X_val=None,
    Y_val=None,
    X_test=None,
    **kwargs,):
    X_all = kwargs["X_all"]
    train_idx = kwargs["train_idx"]

    pred_horizon = kwargs.get("pred_horizon", 24)
    r_col = kwargs.get("r_col", 0)
    n_components = kwargs.get("n_components", 5)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        n_init=20,
        max_iter=1000,
        reg_covar=1e-5,
        random_state=42,)

    gmm.fit(X_train_s)

    train_proba = gmm.predict_proba(X_train_s)

    valid_mask = train_idx <= train_idx[-1] - pred_horizon
    valid_train_idx = train_idx[valid_mask]
    valid_train_proba = train_proba[valid_mask]

    y_fwd_ret = make_future_return_target(
        X_all=X_all,
        end_idx=valid_train_idx,
        horizon=pred_horizon,
        r_col=r_col,)

    regime_score = []

    for k in range(n_components):
        w = valid_train_proba[:, k]
        score = np.sum(w * y_fwd_ret) / np.sum(w)
        regime_score.append(score)

    order = np.argsort(regime_score)

    test_proba = gmm.predict_proba(X_test_s)
    test_proba = test_proba[:, order]

    return [tuple(row) for row in test_proba]

In [48]:
preds_gmm = walk_forward(
    X=X,
    Y=Y,
    dates=dates,
    model=gmm_model,
    tr=5000,
    v=0,
    te=42,
    em=4,
    pred_horizon=24,
    r_col=r_col,
    n_components=5,)
proba = np.vstack(preds_gmm["Y_pred"].to_numpy())
print(proba.shape)

(8148, 5)


In [49]:
proba = np.vstack(preds_gmm["Y_pred"].to_numpy())

regime_probs = pd.DataFrame(
    proba,
    index=preds_gmm.index,
    columns=[f"p_regime_{i}" for i in range(proba.shape[1])],
)

regime_probs.index.name = "datetime"
regime_probs = regime_probs.reset_index()

regime_probs.to_parquet(
    "gmm_regime_probabilities_oos.parquet",
    index=False,
    engine="pyarrow",
)

In [46]:
regime_probs.head()

,datetime,p_regime_0,p_regime_1,p_regime_2,p_regime_3,p_regime_4
0,2022-08-14 20:00:00+00:00,2.142727e-10,0.321036,6.165548e-01,0.061434,0.000975
1,2022-08-15 00:00:00+00:00,2.119117e-09,0.473424,6.228542e-03,0.513554,0.006793
2,2022-08-15 04:00:00+00:00,2.400797e-03,0.861519,3.453041e-07,0.119402,0.016678
3,2022-08-15 08:00:00+00:00,4.244708e-05,0.973680,1.141524e-02,0.014254,0.000608
4,2022-08-15 12:00:00+00:00,8.593604e-06,0.920562,3.715071e-02,0.040677,0.001601


In [47]:
h = 24

res = preds_gmm.copy()
res["regime"] = res["Y_pred"].apply(lambda p: int(np.argmax(p)))
res = res.reset_index().rename(columns={"index": "datetime"})

px = df[["datetime", "close", "r"]].copy()
px["datetime"] = pd.to_datetime(px["datetime"], utc=True)
px = px.sort_values("datetime")

px["fwd_ret"] = px["close"].shift(-h) / px["close"] - 1
px["fwd_log_ret"] = px["r"].rolling(h).sum().shift(-h + 1)

check = res.merge(
    px[["datetime", "r", "fwd_ret", "fwd_log_ret"]],
    on="datetime",
    how="inner",
)

print(
    check.groupby("regime")[["r", "fwd_ret", "fwd_log_ret"]]
         .agg(["count", "mean", "median", "std"])
)

           r                               fwd_ret                      \
       count      mean    median       std   count      mean    median   
regime                                                                   
0       1666  0.000010  0.000269  0.011816    1666  0.006082  0.004800   
1       1616 -0.000529 -0.000153  0.010588    1616 -0.001725 -0.000794   
2       2700 -0.000061  0.000079  0.007795    2700  0.005413  0.001277   
3       1337  0.000318  0.000210  0.009997    1337  0.005753  0.002705   
4        781  0.002221  0.001061  0.010632     781  0.010889  0.005215   

                 fwd_log_ret                                
             std       count      mean    median       std  
regime                                                      
0       0.046958        1666  0.004908  0.004951  0.046692  
1       0.049400        1616 -0.003380 -0.001325  0.049247  
2       0.049701        2700  0.003858  0.001110  0.049220  
3       0.053442        1337  0.004570  0

count    8.280000e+03
mean     1.000000e+00
std      3.022755e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64